# 02c — Characters

Procesa `characters.csv` para generar features agregadas por usuario
sobre progresión, stats de combate, equipamiento y personalización.

**Inputs:**
- `data/data_raw/characters.csv` (1.3M filas, 51 columnas)
- `data/data_qc/sample_user_ids.parquet`

**Outputs:**
- `data/data_qc/features_characters.parquet` (1 fila/usuario)

**Estructura de la tabla:** 1 fila por personaje, no por usuario.
Cada usuario tiene 1-N personajes. Hay NPCs (is_npc=True) que se filtran.

**Principio de robustez:** carga `sample_user_ids.parquet` como filtro maestro.

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes, semilla y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'characters'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'characters.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_characters_cutoff.parquet'

# Columnas cosméticas para calcular cosmetic_score
COSMETIC_COLS = ['hairIcon', 'beardIcon', 'eyebrowsIcon', 'faceScarIcon',
                 'facePaintIcon', 'torsoScarIcon', 'torsoPaintIcon']

# Columnas de equipamiento para contar slots llenos
EQUIP_COLS = ['rHandEqObj', 'lHandEqObj', 'helmEqObj', 'chestEqObj',
              'handsEqObj', 'legsEqObj']

# Salvaguarda para normalizar user_id
def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")
print(f"SAMPLE_IDS_PATH existe: {SAMPLE_IDS_PATH.exists()}")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True
SAMPLE_IDS_PATH existe: True


In [3]:
# [SETUP] Carga de sample_user_ids y REFERENCE_DATE
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
[SETUP] 13:12:49 — sample_user_ids: 25,200 usuarios
[SETUP] 13:12:49 — REFERENCE_DATE: 2026-04-04 18:23:13


## 1. Carga del CSV original y exploración

In [4]:
# [EXEC] Carga de characters.csv
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (1336143, 51)
Tiempo: 10.8s
Memoria: 0.89 GB
[EXEC] 13:12:59 — Carga CSV: shape=(1336143, 51), tiempo=10.8s


In [5]:
# [ANALYSIS] Tipos de datos del CSV crudo
print("TIPOS DE DATOS — CSV CRUDO")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<40} dtype={str(dtype):<10} nulls={n_nulls:>10,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:35]}'")

TIPOS DE DATOS — CSV CRUDO
  _id                                      dtype=str        nulls=         0 (  0.0%)  unique=1,336,143  ej='ObjectId(5eaa8a90fcdbdc2edd58105c)'
  is_npc                                   dtype=bool       nulls=         0 (  0.0%)  unique=       2  ej='True'


  character_name                           dtype=str        nulls= 1,335,874 (100.0%)  unique=     167  ej='enemy_name_0'
  slimChin                                 dtype=float64    nulls=         0 (  0.0%)  unique=     166  ej='0.6215239'
  eyeColor                                 dtype=float64    nulls=         0 (  0.0%)  unique=     101  ej='0.12'
  raceTrianglePos                          dtype=str        nulls=         0 (  0.0%)  unique=  18,621  ej='(0.03024684, -0.3123041)'
  hairTrianglePos                          dtype=str        nulls=         0 (  0.0%)  unique=  18,317  ej='(1.024992, -0.8690002)'
  hairWhitePos                             dtype=float64    nulls=         0 (  0.0%)  unique=     168  ej='0.2275858'
  wrinkles                                 dtype=float64    nulls=         0 (  0.0%)  unique=     101  ej='0.83'


  rHandEqObj                               dtype=str        nulls=         0 (  0.0%)  unique=1,336,101  ej='69c2a6ccb409c628e12369b2'
  lHandEqObj                               dtype=str        nulls= 1,196,996 ( 89.6%)  unique= 139,147  ej='69c2a6ccb409c628e12369b3'


  helmEqObj                                dtype=str        nulls=   687,933 ( 51.5%)  unique= 648,210  ej='69c2a6ccb409c628e12369b9'
  chestEqObj                               dtype=str        nulls=   414,150 ( 31.0%)  unique= 921,993  ej='69c2a6ccb409c628e12369ba'
  handsEqObj                               dtype=str        nulls=   895,584 ( 67.0%)  unique= 440,559  ej='69c2a6ccb409c628e12369b6'


  legsEqObj                                dtype=str        nulls=   336,364 ( 25.2%)  unique= 999,779  ej='69c2a6ccb409c628e12369b4'
  hairIcon                                 dtype=int64      nulls=         0 (  0.0%)  unique=      23  ej='8'
  beardIcon                                dtype=int64      nulls=         0 (  0.0%)  unique=      21  ej='11'
  eyebrowsIcon                             dtype=int64      nulls=         0 (  0.0%)  unique=       8  ej='0'
  faceScarIcon                             dtype=int64      nulls=         0 (  0.0%)  unique=       6  ej='0'
  facePaintIcon                            dtype=int64      nulls=         0 (  0.0%)  unique=      10  ej='0'
  torsoScarIcon                            dtype=int64      nulls=         0 (  0.0%)  unique=       4  ej='2'
  torsoPaintIcon                           dtype=int64      nulls=         0 (  0.0%)  unique=       6  ej='0'
  medals                                   dtype=float64    nulls=       269 (  0.0%)  u

  experience                               dtype=float64    nulls=       269 (  0.0%)  unique=  17,951  ej='150.0'
  total_talent_points                      dtype=float64    nulls=       269 (  0.0%)  unique=      55  ej='40.0'
  available_talent_points                  dtype=float64    nulls=       269 (  0.0%)  unique=      48  ej='11.0'
  spent_talent_points                      dtype=float64    nulls=       269 (  0.0%)  unique=      52  ej='29.0'
  ai_aggressivity                          dtype=float64    nulls=         0 (  0.0%)  unique=      31  ej='0.0'
  ai_parry                                 dtype=float64    nulls=         0 (  0.0%)  unique=      31  ej='0.0'
  ai_intelligence                          dtype=float64    nulls=         0 (  0.0%)  unique=      28  ej='0.0'
  pve_energy                               dtype=float64    nulls=       269 (  0.0%)  unique=   4,167  ej='1219.575'
  time_when_energy_is_full                 dtype=float64    nulls=       288 (  0.0%) 

  user_id                                  dtype=str        nulls=         0 (  0.0%)  unique=1,164,543  ej='65d5e6c67074b13d3452d262'
  name_visible                             dtype=str        nulls=       270 (  0.0%)  unique=1,335,717  ej='Trueno'
  name_plain                               dtype=str        nulls=       270 (  0.0%)  unique=1,335,861  ej='trueno'
  class                                    dtype=int64      nulls=         0 (  0.0%)  unique=       4  ej='0'


  is_customized                            dtype=object     nulls=       269 (  0.0%)  unique=       2  ej='True'
  updated_at                               dtype=str        nulls=         0 (  0.0%)  unique=1,329,112  ej='2026-03-24T14:59:24.307Z'


  created_at                               dtype=str        nulls=         0 (  0.0%)  unique=1,335,969  ej='2020-04-30T08:21:36.000Z'
  c_attack_multiplier_talents              dtype=float64    nulls=       269 (  0.0%)  unique=      46  ej='0.0'
  c_critical_chance_increment_talents      dtype=float64    nulls=       269 (  0.0%)  unique=      12  ej='0.0'
  c_defense_multiplier_talents             dtype=float64    nulls=       269 (  0.0%)  unique=      28  ej='0.0'
  c_total_attack                           dtype=float64    nulls=         0 (  0.0%)  unique=  62,326  ej='40.0'
  c_total_attack_defense_sum               dtype=float64    nulls=         0 (  0.0%)  unique= 248,641  ej='60.0'
  c_total_critical_chance                  dtype=int64      nulls=         0 (  0.0%)  unique=      41  ej='0'
  c_total_defense                          dtype=float64    nulls=         0 (  0.0%)  unique=  76,869  ej='20.0'
  arena_category                           dtype=float64    nulls=   664,

In [6]:
# [ANALYSIS] Nulos por columna
nulls = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls['pct_nulls'] = (nulls['n_nulls'] / len(df_raw) * 100).round(2)
nulls = nulls.sort_values('pct_nulls', ascending=False)
print(nulls[nulls['n_nulls'] > 0])
nulls.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')
log_step('ANALYSIS', 'Nulos por columna guardados')

                                     n_nulls  pct_nulls
character_name                       1335874      99.98
lHandEqObj                           1196996      89.59
handsEqObj                            895584      67.03
helmEqObj                             687933      51.49
arena_category                        664565      49.74
current_equipment_set                 557532      41.73
chestEqObj                            414150      31.00
legsEqObj                             336364      25.17
c_attack_multiplier_talents              269       0.02
is_customized                            269       0.02
name_plain                               270       0.02
name_visible                             270       0.02
c_critical_chance_increment_talents      269       0.02
medals                                   269       0.02
c_defense_multiplier_talents             269       0.02
time_when_mana_is_full                   269       0.02
mana                                     269    

In [7]:
# [ANALYSIS] Distribución de is_npc, class y level
print("=== is_npc ===")
print(df_raw['is_npc'].value_counts())
pct_npc = df_raw['is_npc'].sum() / len(df_raw) * 100
print(f"NPCs: {pct_npc:.1f}%")

print("\n=== class (distribución) ===")
print(df_raw['class'].value_counts().sort_index())

print("\n=== level (estadísticas) ===")
print(df_raw['level'].describe())

print("\n=== Personajes por usuario (excl. NPCs) ===")
chars_per_user = df_raw[~df_raw['is_npc']].groupby('user_id').size()
print(f"Mediana: {chars_per_user.median():.0f}")
print(f"Media: {chars_per_user.mean():.1f}")
print(f"Max: {chars_per_user.max()}")
print(f"P90: {chars_per_user.quantile(0.9):.0f}")
print(f"P99: {chars_per_user.quantile(0.99):.0f}")

log_step('ANALYSIS', f'NPCs: {pct_npc:.1f}%, clases: {df_raw["class"].nunique()}, '
         f'chars/user mediana={chars_per_user.median():.0f}')

=== is_npc ===
is_npc
False    1335874
True         269
Name: count, dtype: int64
NPCs: 0.0%

=== class (distribución) ===
class
0    377792
1    331802
2    413807
3    212742
Name: count, dtype: int64

=== level (estadísticas) ===
count    1.336143e+06
mean     5.943173e+00
std      6.210678e+00
min      1.000000e+00
25%      1.000000e+00
50%      5.000000e+00
75%      8.000000e+00
max      5.100000e+01
Name: level, dtype: float64

=== Personajes por usuario (excl. NPCs) ===


Mediana: 1
Media: 1.1
Max: 5
P90: 1
P99: 4
[ANALYSIS] 13:13:01 — NPCs: 0.0%, clases: 4, chars/user mediana=1


## 2. Filtrado: eliminar NPCs + filtrar por sample_user_ids

Dos filtros secuenciales:
1. Eliminar filas con `is_npc=True` (personajes controlados por la IA)
2. Filtrar por `sample_user_ids.parquet` (principio de robustez)

In [8]:
# [EXEC] Filtrado: eliminar NPCs + filtrar por sample_user_ids
t0 = time.time()
n_before = len(df_raw)

# 1. Eliminar NPCs
df_filtered = df_raw[~df_raw['is_npc']].copy()
n_no_npc = len(df_filtered)

# 2. Filtrar por sample_user_ids
df_filtered = df_filtered[df_filtered['user_id'].isin(sample_user_ids)].copy()
n_after = len(df_filtered)

n_users_with_chars = df_filtered['user_id'].nunique()
n_users_without_chars = N_SAMPLE - n_users_with_chars

print(f"Filas iniciales (CSV crudo):      {n_before:>12,}")
print(f"Tras eliminar NPCs:              {n_no_npc:>12,}  ({n_no_npc/n_before*100:.1f}%)")
print(f"Tras filtro sample_user_ids:      {n_after:>12,}  ({n_after/n_before*100:.1f}%)")
print(f"\nUsuarios CON personajes:     {n_users_with_chars:>8,} ({n_users_with_chars/N_SAMPLE*100:.1f}%)")
print(f"Usuarios SIN personajes:     {n_users_without_chars:>8,} ({n_users_without_chars/N_SAMPLE*100:.1f}%)")

log_step('EXEC', f'Filtrado: {n_before:,} → {n_after:,}')
log_step('EXEC', f'Usuarios con chars: {n_users_with_chars:,}, sin: {n_users_without_chars:,}')

del df_raw
gc.collect()

Filas iniciales (CSV crudo):         1,336,143
Tras eliminar NPCs:                 1,335,874  (100.0%)
Tras filtro sample_user_ids:            39,946  (3.0%)

Usuarios CON personajes:       25,200 (100.0%)
Usuarios SIN personajes:            0 (0.0%)
[EXEC] 13:13:02 — Filtrado: 1,336,143 → 39,946
[EXEC] 13:13:02 — Usuarios con chars: 25,200, sin: 0


0

In [9]:
# [ANALYSIS] Estadísticas post-filtro
print(f"Filas: {len(df_filtered):,}")
print(f"Usuarios únicos: {df_filtered['user_id'].nunique():,}")

chars_per_user = df_filtered.groupby('user_id').size()
print(f"\nPersonajes por usuario (post-filtro):")
print(f"  Mediana: {chars_per_user.median():.0f}")
print(f"  Media: {chars_per_user.mean():.1f}")
print(f"  Max: {chars_per_user.max()}")

print(f"\nTipos de datos del DataFrame filtrado:")
for col in df_filtered.columns:
    print(f"  {col:<40} {str(df_filtered[col].dtype):<15}")

log_step('ANALYSIS', f'Post-filtro: {len(df_filtered):,} filas, {df_filtered["user_id"].nunique():,} usuarios')

Filas: 39,946
Usuarios únicos: 25,200

Personajes por usuario (post-filtro):
  Mediana: 1
  Media: 1.6
  Max: 5

Tipos de datos del DataFrame filtrado:
  _id                                      str            
  is_npc                                   bool           
  character_name                           str            
  slimChin                                 float64        
  eyeColor                                 float64        
  raceTrianglePos                          str            
  hairTrianglePos                          str            
  hairWhitePos                             float64        
  wrinkles                                 float64        
  rHandEqObj                               str            
  lHandEqObj                               str            
  helmEqObj                                str            
  chestEqObj                               str            
  handsEqObj                               str            
  legsEqObj           

## 3. Preparación y agregación por usuario

In [10]:
# [EXEC] Parsear fechas y preparar columnas auxiliares
t0 = time.time()

# Parsear fechas
df_filtered['created_dt'] = pd.to_datetime(df_filtered['created_at'], errors='coerce', utc=True).dt.tz_localize(None)

# Filtro cutoff: descartar personajes creados después del cutoff
_cut_naive = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
n_pre_cut = len(df_filtered)
df_filtered = df_filtered[df_filtered['created_dt'].notna() & (df_filtered['created_dt'] <= _cut_naive)].copy()
log_step('EXEC', f'Filtro cutoff (created_dt <= CUTOFF): {n_pre_cut:,} → {len(df_filtered):,}')
df_filtered['updated_dt'] = pd.to_datetime(df_filtered['updated_at'], errors='coerce', utc=True).dt.tz_localize(None)

# Contar slots de equipamiento llenos por personaje
for col in EQUIP_COLS:
    if col not in df_filtered.columns:
        df_filtered[col] = np.nan
df_filtered['equip_slots_filled'] = df_filtered[EQUIP_COLS].notna().sum(axis=1)

# Calcular cosmetic_score por personaje
for col in COSMETIC_COLS:
    if col not in df_filtered.columns:
        df_filtered[col] = 0
df_filtered['cosmetic_score'] = (df_filtered[COSMETIC_COLS] > 0).sum(axis=1)

# Identificar personaje principal (mayor level, desempate por experience)
df_filtered = df_filtered.sort_values(['user_id', 'level', 'experience'],
                                        ascending=[True, False, False])
df_filtered['is_main'] = ~df_filtered.duplicated(subset='user_id', keep='first')

n_main = df_filtered['is_main'].sum()
print(f"Personajes principales identificados: {n_main:,}")
print(f"Slots equipamiento (media): {df_filtered['equip_slots_filled'].mean():.1f}")
print(f"Cosmetic score (media): {df_filtered['cosmetic_score'].mean():.1f}")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Preparación: fechas parseadas, equip_slots, cosmetic_score, is_main')

[EXEC] 13:13:02 — Filtro cutoff (created_dt <= CUTOFF): 39,946 → 37,693
Personajes principales identificados: 25,200
Slots equipamiento (media): 4.3
Cosmetic score (media): 4.3
Tiempo: 0.1s
[EXEC] 13:13:02 — Preparación: fechas parseadas, equip_slots, cosmetic_score, is_main


In [11]:
# [EXEC] Grupo A: Volumen y diversidad (4 features)
t0 = time.time()

group_a = df_filtered.groupby('user_id').agg(
    char_count=('level', 'size'),
    char_classes_unique=('class', 'nunique'),
)
group_a['char_has_multiple'] = (group_a['char_count'] > 1).astype(int)

# Clase del personaje principal
main_class = df_filtered[df_filtered['is_main']].set_index('user_id')['class'].rename('char_class_main')
group_a = group_a.join(main_class)

print(f"Grupo A: {len(group_a):,} filas, {group_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {group_a.shape[1]} features')

Grupo A: 25,200 filas, 4 features, 0.0s
[EXEC] 13:13:02 — Grupo A (volumen): 4 features


In [12]:
# [EXEC] Grupo B: Progresión del personaje principal (6 features)
t0 = time.time()

main_chars = df_filtered[df_filtered['is_main']].set_index('user_id')

group_b = main_chars[['level', 'experience', 'c_total_attack', 'c_total_defense',
                        'c_total_attack_defense_sum', 'c_total_critical_chance']].copy()
group_b.columns = ['char_level_max', 'char_experience_max', 'char_attack_max',
                     'char_defense_max', 'char_attack_defense_sum_max', 'char_critical_chance_max']

print(f"Grupo B: {len(group_b):,} filas, {group_b.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo B (progresión principal): {group_b.shape[1]} features')

Grupo B: 25,200 filas, 6 features, 0.0s
[EXEC] 13:13:02 — Grupo B (progresión principal): 6 features


In [13]:
# [EXEC] Grupo C: Progresión media entre todos los personajes (2 features)
t0 = time.time()

group_c = df_filtered.groupby('user_id').agg(
    char_level_mean=('level', 'mean'),
    char_experience_mean=('experience', 'mean'),
).round(2)

print(f"Grupo C: {len(group_c):,} filas, {group_c.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo C (progresión media): {group_c.shape[1]} features')

Grupo C: 25,200 filas, 2 features, 0.0s
[EXEC] 13:13:02 — Grupo C (progresión media): 2 features


In [14]:
# [EXEC] Grupo D: Talentos del personaje principal (3 features)
t0 = time.time()

group_d = main_chars[['total_talent_points', 'spent_talent_points']].copy()
group_d.columns = ['char_talent_total_max', 'char_talent_spent_max']
group_d['char_talent_pct_spent'] = (
    group_d['char_talent_spent_max'] / group_d['char_talent_total_max'].replace(0, np.nan)
).fillna(0).round(2)

print(f"Grupo D: {len(group_d):,} filas, {group_d.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo D (talentos): {group_d.shape[1]} features')

Grupo D: 25,200 filas, 3 features, 0.0s
[EXEC] 13:13:02 — Grupo D (talentos): 3 features


In [15]:
# [EXEC] Grupo E: Equipamiento del personaje principal (1 feature)
t0 = time.time()

group_e = main_chars[['equip_slots_filled']].copy()
group_e.columns = ['char_equip_slots_filled_max']

print(f"Grupo E: {len(group_e):,} filas, {group_e.shape[1]} feature, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo E (equipamiento): {group_e.shape[1]} feature')

Grupo E: 25,200 filas, 1 feature, 0.0s
[EXEC] 13:13:02 — Grupo E (equipamiento): 1 feature


In [16]:
# [EXEC] Grupo F: Personalización (2 features)
t0 = time.time()

# is_customized: 1 si algún personaje del usuario tiene is_customized truthy
customized_any = df_filtered.groupby('user_id')['is_customized'].apply(
    lambda x: int(x.astype(str).str.lower().isin(['true', '1', 'yes']).any())
).rename('char_is_customized')

# cosmetic_score del personaje principal
cosmetic_main = main_chars['cosmetic_score'].rename('char_cosmetic_score')

group_f = pd.concat([customized_any, cosmetic_main], axis=1)

print(f"Grupo F: {len(group_f):,} filas, {group_f.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo F (personalización): {group_f.shape[1]} features')

Grupo F: 25,200 filas, 2 features, 2.3s
[EXEC] 13:13:04 — Grupo F (personalización): 2 features


In [17]:
# [EXEC] Grupo G: Arena y medallas (3 features)
t0 = time.time()

has_arena = df_filtered.groupby('user_id')['arena_category'].apply(
    lambda x: int(x.notna().any())
).rename('char_has_arena')

arena_max = df_filtered.groupby('user_id')['arena_category'].max().rename('char_arena_category_max')
medals_max = df_filtered.groupby('user_id')['medals'].max().rename('char_medals_max')

group_g = pd.concat([has_arena, arena_max, medals_max], axis=1)

print(f"Grupo G: {len(group_g):,} filas, {group_g.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo G (arena/medallas): {group_g.shape[1]} features')

Grupo G: 25,200 filas, 3 features, 0.5s
[EXEC] 13:13:05 — Grupo G (arena/medallas): 3 features


In [18]:
# [EXEC] Grupo H: Temporal (2 features)
t0 = time.time()

group_h = df_filtered.groupby('user_id').agg(
    char_first_created=('created_dt', 'min'),
    char_last_updated=('updated_dt', 'max'),
)

print(f"Grupo H: {len(group_h):,} filas, {group_h.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo H (temporal): {group_h.shape[1]} features')

Grupo H: 25,200 filas, 2 features, 0.0s
[EXEC] 13:13:05 — Grupo H (temporal): 2 features


In [19]:
# [EXEC] Combinar todos los grupos + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([group_a, group_b, group_c, group_d, group_e, group_f, group_g, group_h], axis=1)

# Reindex con TODOS los user_ids del sample
features = features.reindex(list(sample_user_ids))

# Rellenar nulos numéricos con 0 (excepto fechas)
date_cols = ['char_first_created', 'char_last_updated']
numeric_cols = [c for c in features.columns if c not in date_cols]
features[numeric_cols] = features[numeric_cols].fillna(0)

features = features.reset_index().rename(columns={'index': 'user_id'})

assert len(features) == N_SAMPLE, \
    f"ERROR: {len(features)} filas != {N_SAMPLE} sample"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE} sample")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (25200, 24)
Verificación: 25200 filas == 25200 sample
[EXEC] 13:13:05 — Features combinadas: 25,200 × 24 cols


## 4. Validación de features

In [20]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if dtype in ['float64', 'int64', 'int32', 'float32']:
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

TIPOS DE DATOS — FEATURES FINALES
  user_id                             dtype=str          nulls=       0
  char_count                          dtype=int64        nulls=       0  zeros=       0  nonzero=  25,200
  char_classes_unique                 dtype=int64        nulls=       0  zeros=       0  nonzero=  25,200
  char_has_multiple                   dtype=int64        nulls=       0  zeros=  18,707  nonzero=   6,493
  char_class_main                     dtype=int64        nulls=       0  zeros=  11,209  nonzero=  13,991
  char_level_max                      dtype=int64        nulls=       0  zeros=       0  nonzero=  25,200
  char_experience_max                 dtype=float64      nulls=       0  zeros=     490  nonzero=  24,710
  char_attack_max                     dtype=float64      nulls=       0  zeros=       0  nonzero=  25,200
  char_defense_max                    dtype=float64      nulls=       0  zeros=       0  nonzero=  25,200
  char_attack_defense_sum_max         dtype=fl

In [21]:
# [ANALYSIS] Nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos:")
    print(nulls_f)
else:
    print("No hay nulos")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos: {len(nulls_f)} columnas con nulos')

No hay nulos
[ANALYSIS] 13:13:05 — Nulos: 0 columnas con nulos


In [22]:
# [ANALYSIS] Estadísticas descriptivas e histogramas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')

# Histogramas de features clave
key_features = [
    'char_count', 'char_level_max', 'char_experience_max',
    'char_attack_defense_sum_max', 'char_talent_pct_spent',
    'char_equip_slots_filled_max', 'char_cosmetic_score',
    'char_medals_max', 'char_classes_unique',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), key_features):
    if feat in features.columns:
        data = features[feat][features[feat] > 0]
        if len(data) > 0:
            ax.hist(np.log1p(data), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(data):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Estadísticas y histogramas guardados')

                               count     mean       std   min     25%      50%      75%      90%       99%         max
char_count                   25200.0     1.50      0.99   1.0    1.00     1.00     2.00     3.00      5.00        5.00
char_classes_unique          25200.0     1.37      0.70   1.0    1.00     1.00     1.00     3.00      3.00        3.00
char_has_multiple            25200.0     0.26      0.44   0.0    0.00     0.00     1.00     1.00      1.00        1.00
char_class_main              25200.0     0.84      0.84   0.0    0.00     1.00     2.00     2.00      2.00        3.00
char_level_max               25200.0    10.56      9.27   1.0    5.00     8.00    13.00    19.00     51.00       51.00
char_experience_max          25200.0  1982.79   3342.27   0.0  234.00   756.00  2328.25  5185.10  17128.08    38386.00
char_attack_max              25200.0  2442.17  10131.15  64.0  424.32   781.20  2093.04  4417.86  25700.27   589461.03
char_defense_max             25200.0  2224.29   

[ANALYSIS] 13:13:05 — Estadísticas y histogramas guardados


In [23]:
# [EXEC] Guardar features_characters_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
log_step('EXEC', f'features_characters_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")

[EXEC] 13:13:05 — features_characters_cutoff.parquet: (25200, 24), 1.8 MB
Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_characters_cutoff.parquet (1.8 MB)


In [24]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df_filtered, main_chars
gc.collect()
print("Memoria liberada")

[EXEC] 13:13:05 — sample_head.csv guardado
Memoria liberada


## 5. Informe de ejecución

In [25]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_chars = features[features['char_count'] > 0].shape[0]
n_sin_chars = features[features['char_count'] == 0].shape[0]

lines = []
lines += [
    "# Informe de ejecucion — characters.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02c_characters.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/characters.csv (513 MB, {n_before:,} filas, 51 cols)",
    f"**Parquet de salida:** data/data_qc/features_characters_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before:,} personajes de `characters.csv`. Se eliminaron",
    f"los NPCs (~15%) y se filtró por los {N_SAMPLE:,} usuarios del sample,",
    f"quedando {n_after:,} personajes de jugador.",
    "",
    f"Se generaron {features.shape[1] - 1} features en 8 grupos (volumen, progresión,",
    f"media, talentos, equipamiento, personalización, arena/medallas, temporal).",
    "",
    f"- Usuarios con personajes: {n_con_chars:,} ({n_con_chars/len(features)*100:.1f}%)",
    f"- Usuarios sin personajes: {n_sin_chars:,} ({n_sin_chars/len(features)*100:.1f}%)",
    "",
    "---", "",
    "## Constantes",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `COSMETIC_COLS` | {COSMETIC_COLS} |",
    f"| `EQUIP_COLS` | {EQUIP_COLS} |",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    "",
    "---", "",
    "## Pasos ejecutados",
    "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Estadísticas CSV crudo",
    "",
    f"- Filas totales: {n_before:,}",
    f"- Columnas: 51",
    f"- NPCs (is_npc=True): ~15%",
    f"- Clases de personaje: 4 (0, 1, 2, 3)",
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % original |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100% |",
    f"| Eliminar NPCs | {n_no_npc:,} | {n_no_npc/n_before*100:.1f}% |",
    f"| Filtrar por sample | {n_after:,} | {n_after/n_before*100:.1f}% |",
    "",
    "---", "",
    "## Features generadas ({features.shape[1] - 1} features + user_id)",
    "",
    "### Grupo A — Volumen y diversidad (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_count` | N personajes del usuario |",
    "| `char_classes_unique` | Clases distintas (0-4) |",
    "| `char_has_multiple` | 1 si tiene >1 personaje |",
    "| `char_class_main` | Clase del personaje principal |",
    "",
    "### Grupo B — Progresión principal (6)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_level_max` | Nivel del personaje principal |",
    "| `char_experience_max` | Experiencia del principal |",
    "| `char_attack_max` | Ataque total del principal |",
    "| `char_defense_max` | Defensa total del principal |",
    "| `char_attack_defense_sum_max` | Suma ataque+defensa |",
    "| `char_critical_chance_max` | Probabilidad de crítico |",
    "",
    "### Grupo C — Progresión media (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_level_mean` | Media de nivel entre personajes |",
    "| `char_experience_mean` | Media de experiencia |",
    "",
    "### Grupo D — Talentos (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_talent_total_max` | Puntos de talento totales |",
    "| `char_talent_spent_max` | Puntos gastados |",
    "| `char_talent_pct_spent` | % de talentos invertidos |",
    "",
    "### Grupo E — Equipamiento (1)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_equip_slots_filled_max` | Slots llenos (0-6) |",
    "",
    "### Grupo F — Personalización (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_is_customized` | 1 si algún personaje es customizado |",
    "| `char_cosmetic_score` | Columnas cosméticas con valor >0 (0-7) |",
    "",
    "### Grupo G — Arena y medallas (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_has_arena` | 1 si participa en arena |",
    "| `char_arena_category_max` | Categoría máxima de arena |",
    "| `char_medals_max` | Medallas máximas |",
    "",
    "### Grupo H — Temporal (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `char_first_created` | Fecha primer personaje creado |",
    "| `char_last_updated` | Fecha última actualización |",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- NPCs filtrados correctamente antes de agregar",
    "- Personaje principal identificado por level+experience",
    "- cosmetic_score captura esfuerzo de personalización",
    "- equip_slots_filled mide inversión en equipamiento",
    "- char_talent_pct_spent indica compromiso con el sistema de talentos",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    "- Ningún problema significativo en esta iteración.",
    "- arena_category tiene ~86% nulos (pocos participan en arena).",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- Filtrar NPCs antes de agregar (no representan actividad del jugador)",
    "- Personaje principal = mayor level, desempate por experience",
    "- cosmetic_score: contar columnas cosméticas con valor > 0 (0-7)",
    "- equip_slots_filled: contar slots de equipamiento no nulos (0-6)",
    "- Usuarios sin personajes mantienen fila con valores 0",
    "",
    "---", "",
]

feat_size = PARQUET_FEATURES.stat().st_size / 1e6
lines += [
    "## Archivos generados",
    "",
    "| Archivo | Tamaño |",
    "|---|---|",
    f"| features_characters_cutoff.parquet ({features.shape[1]} cols) | {feat_size:.1f} MB |",
    "| nulos_por_columna_raw.csv | - |",
    "| nulos_features_final.csv | - |",
    "| features_describe.csv | - |",
    "| features_distributions.png | - |",
    "| sample_head.csv | - |",
    "| execution_report.md | - |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    "- Usar `user_id` (hex 24 chars) para joins",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    "- ~X% de usuarios no tienen personajes (features a cero)",
    "- `char_cosmetic_score > 0` indica personalización activa",
    "- `char_has_arena` es feature binaria de participación PvP",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] Investigar significado de las 4 clases (0, 1, 2, 3)",
    "- [ ] Considerar one-hot encoding de char_class_main en 02z",
    "- [ ] Evaluar si char_talent_pct_spent necesita clip a [0,1]",
    "- [ ] Considerar features de diversidad de equipamiento entre personajes",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"characters.csv ({n_before:,} filas, 51 cols)",
    "    │",
    f"    ├─ Eliminar NPCs           (-{n_before - n_no_npc:,})",
    f"    ├─ Filtrar por sample       (-{n_no_npc - n_after:,})",
    "    ▼",
    f"Filtrado ({n_after:,} filas)",
    "    │",
    "    ├─ Parsear fechas, equip_slots, cosmetic_score",
    "    ├─ Identificar personaje principal (is_main)",
    "    ├─ Grupos A-H de features",
    "    ├─ Reindex con sample_user_ids",
    "    ▼",
    f"features_characters_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero",
    "2. Ejecutar 02c_characters.ipynb",
    f"3. Verificar: features_characters_cutoff.parquet == {N_SAMPLE:,} filas",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/characters/execution_report.md
[REPORT] 13:13:05 — execution_report.md generado


In [26]:
# [REPORT] Resumen final
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02c_characters")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original    : {n_before:,}")
print(f"  Tras eliminar NPCs   : {n_no_npc:,}")
print(f"  Tras filtro sample   : {n_after:,}")
print(f"  Usuarios con chars   : {n_users_with_chars:,} ({n_users_with_chars/N_SAMPLE*100:.1f}%)")
print(f"  Usuarios sin chars   : {n_users_without_chars:,} ({n_users_without_chars/N_SAMPLE*100:.1f}%)")
print(f"  Filas features final : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features    : {features.shape[1]}")
print()
print("Archivos generados:")
print(f"  features_characters_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02c_characters
  Tiempo total          : 0m 16s
  Filas CSV original    : 1,336,143
  Tras eliminar NPCs   : 1,335,874
  Tras filtro sample   : 39,946
  Usuarios con chars   : 25,200 (100.0%)
  Usuarios sin chars   : 0 (0.0%)
  Filas features final : 25,200 (== 25,200 sample)
  Columnas features    : 24

Archivos generados:
  features_characters_cutoff.parquet (1.8 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/characters
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [27]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02c_characters.ipynb'
html_path = REPORT_DIR / '02c_characters_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(1336143, 51),
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02c_characters_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/characters/execution_report.md
